# Run the initial FD001 detector pilot. Final results are completed in 05B.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, time
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
    precision_score, recall_score, f1_score, confusion_matrix)


DATA_DIR = Path("data")
PROTOCOL_DIR = Path("revision_outputs") / "02_FD001_protocol"
OUTPUT_ROOT = Path("revision_outputs") / "03_FD001_raw_pilot"
FOLD = 1
SEED = 42
SENSORS = [f"Sensor{i}" for i in range(1,22)]
COLUMNS = ["UnitNumber", "Time", "OpSet1", "OpSet2", "OpSet3"] + SENSORS
EXPECTED_PROTOCOL = "880c712efeb9fdbdeec4d940c24247f7b1bab5c69a00e098106f88fc9c6774e8"
CANDIDATES = {"IF": [{"n_estimators":n} for n in [100,200,400]],
              "LOF": [{"n_neighbors":n} for n in [20,50,100]],
              "kNN": [{"n_neighbors":n} for n in [20,50,100]]}


In [ ]:
def require(ok, message):
    if not bool(ok): raise ValueError(message)

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def choose_threshold(y, scores):
    require(np.isfinite(scores).all() and len(np.unique(y))==2, "Geçersiz seçim skorları/etiketleri.")
    p,r,t=precision_recall_curve(y, scores)
    f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(t),where=(p[:-1]+r[:-1])>0)

    i=np.flatnonzero(np.isclose(f,f.max(),rtol=0,atol=1e-12))[-1]
    return float(t[i])

def fit_candidate(method, params, X):
    if method=="IF":
        model=IsolationForest(**params,max_samples="auto",contamination="auto",random_state=SEED,n_jobs=-1)
    elif method=="LOF":
        require(params["n_neighbors"]<len(X), "LOF için yetersiz fit örneği.")
        model=LocalOutlierFactor(**params,novelty=True,contamination="auto",metric="euclidean",n_jobs=-1)
    else:
        require(params["n_neighbors"]<=len(X), "kNN için yetersiz fit örneği.")
        model=NearestNeighbors(**params,metric="euclidean",n_jobs=-1)
    return model.fit(X)

def score(method, model, X):


    values = (model.kneighbors(X,return_distance=True)[0][:,-1]
              if method=="kNN" else -model.score_samples(X))
    require(np.isfinite(values).all(), "Sonlu olmayan model skoru.")
    return values

def metrics(y,scores,threshold):
    pred=(scores>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {"AP":float(average_precision_score(y,scores)),
            "ROC_AUC":float(roc_auc_score(y,scores)),
            "precision":float(precision_score(y,pred,zero_division=0)),
            "recall":float(recall_score(y,pred,zero_division=0)),
            "F1":float(f1_score(y,pred,zero_division=0)),
            "Macro_F1":float(f1_score(y,pred,average="macro",zero_division=0)),
            "TP":int(tp),"FP":int(fp),"TN":int(tn),"FN":int(fn)}


In [ ]:
summary_path=PROTOCOL_DIR/"protocol_summary.json"
require(summary_path.is_file(), f"Dosya yok: {summary_path}")
stage02=json.loads(summary_path.read_text(encoding="utf-8"))
require(stage02["status"]=="PASS" and stage02["canonical_protocol_sha256"]==EXPECTED_PROTOCOL,
        "Kontrol edilen 02 protokolüyle uyuşmuyor.")
for name, expected in stage02["output_file_sha256"].items():
    require((PROTOCOL_DIR/name).is_file() and digest(PROTOCOL_DIR/name)==expected,
            f"02 çıktısı eksik/değişmiş: {name}")
protocol=json.loads((PROTOCOL_DIR/"experiment_protocol.json").read_text(encoding="utf-8"))
canonical=json.dumps(protocol,sort_keys=True,ensure_ascii=False,separators=(",",":"))
require(hashlib.sha256(canonical.encode()).hexdigest()==EXPECTED_PROTOCOL,"Protokol içeriği farklı.")
train_path=DATA_DIR/"train_FD001.txt"
require(train_path.is_file(),f"Dosya yok: {train_path}")
require(digest(train_path)==protocol["source_raw_file_sha256_from_stage01"]["train_FD001.txt"],
        "Ham eğitim dosyası 01 aşamasındaki dosyadan farklı.")
raw=pd.read_csv(train_path,sep=r"\s+",header=None)
require(raw.shape==(20631,26),"Ham veri boyutu farklı.")
raw.columns=COLUMNS
require(np.isfinite(raw.to_numpy(dtype=float)).all(),"Ham veride eksik/sonsuz değer.")
reference=pd.read_csv(PROTOCOL_DIR/"train_evaluation_reference.csv")
require(len(reference)==len(raw) and np.array_equal(reference.source_row,np.arange(len(raw))),"Metadata sırası farklı.")
require(np.array_equal(raw[["UnitNumber","Time"]].to_numpy(),reference[["UnitNumber","Time"]].to_numpy()),
        "Sensör ve metadata satırları eşleşmiyor.")
require(np.array_equal(reference.late_life_le_30,reference.RUL.le(30).astype(int)),"Referans etiketi farklı.")
splits=pd.read_csv(PROTOCOL_DIR/"engine_splits.csv")
fold=splits.loc[splits.scheme.eq("cv") & splits.fold.eq(FOLD)].copy()
require(fold.engine_key.is_unique and len(fold)==100,"Motor rol çakışması/eksik motor.")
require(fold.role.value_counts().to_dict()=={"fit":70,"early_stop":10,"selection":10,"outer_valid":10},
        "Beklenen motor sayıları farklı.")
role_map=fold.set_index("engine_key").role
roles=reference.engine_key.map(role_map)
require(roles.notna().all(),"Atanmamış motor var.")
masks={role:roles.eq(role).to_numpy() for role in ["fit","early_stop","selection","outer_valid"]}
X=raw[SENSORS].to_numpy(dtype=np.float32)
scaler=StandardScaler().fit(X[masks["fit"]])
Xs={role:scaler.transform(X[mask]) for role,mask in masks.items() if role!="early_stop"}
y_selection=reference.loc[masks["selection"],"late_life_le_30"].to_numpy(dtype=int)
require(len(np.unique(y_selection))==2,"Seçim kümesi tek sınıf.")


run_id=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUT=OUTPUT_ROOT/run_id
OUT.mkdir(parents=True,exist_ok=False)
plan={"stage":"03_raw_pilot","fold":FOLD,"seed":SEED,"candidates":CANDIDATES,
      "representation":"StandardScaler on fit engines; all 21 sensors",
      "selection":"AP; ties keep first candidate in listed order",
      "threshold":"positive F1 on selection engines; ties choose higher threshold; score >= threshold",
      "score_definitions":{"IF":"negative score_samples","LOF":"negative novelty score_samples",
                           "kNN":"kth Euclidean distance to fit samples"},
      "protocol_sha256":EXPECTED_PROTOCOL,"raw_train_sha256":digest(train_path)}
(OUT/"pilot_plan.json").write_text(json.dumps(plan,indent=2),encoding="utf-8")
print("Girdi kontrolü tamamlandı. Çıktı klasörü:",OUT.resolve())


In [ ]:

chosen={}
trials=[]
for method,candidates in CANDIDATES.items():
    best=None
    for i,params in enumerate(candidates):
        started=time.perf_counter()
        model=fit_candidate(method,params,Xs["fit"])
        selection_scores=score(method,model,Xs["selection"])
        ap=float(average_precision_score(y_selection,selection_scores))
        trials.append({"method":method,"candidate_order":i,"params":params,
                       "selection_AP":ap,"seconds":time.perf_counter()-started})
        if best is None or ap>best["selection_AP"]:
            best={"model":model,"params":params,"selection_AP":ap,"scores":selection_scores}
        print(method,params,"selection AP =",round(ap,6),flush=True)
    best["threshold"]=choose_threshold(y_selection,best["scores"])
    chosen[method]=best
(OUT/"selection_trials.json").write_text(json.dumps(trials,indent=2),encoding="utf-8")
print("Seçimler tamamlandı. Şimdi dış fold değerlendiriliyor.")


In [ ]:
held=reference.loc[masks["outer_valid"]].reset_index(drop=True)
results=[]
prediction_frames=[]
for method,best in chosen.items():
    scores=score(method,best["model"],Xs["outer_valid"])
    for window in [20,30,40]:
        y=held[f"late_life_le_{window}"].to_numpy(dtype=int)
        require(len(np.unique(y))==2,"Dış değerlendirme referansı tek sınıf.")
        results.append({"method":method,"RUL_window":window,"selected_params":best["params"],
                        "selection_AP_primary":best["selection_AP"],"threshold":best["threshold"],
                        **metrics(y,scores,best["threshold"])})
    pred=held[["sample_key","engine_key","UnitNumber","Time","RUL",
               "late_life_le_20","late_life_le_30","late_life_le_40"]].copy()
    pred["method"]=method
    pred["score"]=scores
    pred["threshold"]=best["threshold"]
    pred["prediction"]=(scores>=best["threshold"]).astype(int)
    prediction_frames.append(pred)
pd.concat(prediction_frames,ignore_index=True).to_csv(OUT/"outer_predictions.csv",index=False)
pd.DataFrame(results).to_csv(OUT/"pilot_metrics.csv",index=False)
(OUT/"scaler_parameters.json").write_text(json.dumps({"features":SENSORS,"mean":scaler.mean_.tolist(),
    "scale":scaler.scale_.tolist(),"fit_rows":int(masks["fit"].sum())},indent=2),encoding="utf-8")
summary={"stage":"03_raw_pilot","status":"PASS","scope":"one outer fold; raw sensor baseline only",
         "fold":FOLD,"seed":SEED,"protocol_sha256":EXPECTED_PROTOCOL,
         "versions":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,
                     "sklearn":sklearn.__version__},
         "roles":{role:{"engines":int(reference.loc[mask,"engine_key"].nunique()),"rows":int(mask.sum())}
                  for role,mask in masks.items()},
         "test_files_read":False,"AE_trained":False,"model_selection_reference":"RUL <= 30 proxy",
         "outer_primary_prevalence":float(held.late_life_le_30.mean()),
         "primary_results":[r for r in results if r["RUL_window"]==30],
         "sensitivity_results":[r for r in results if r["RUL_window"]!=30],
         "selection_trials":trials,
         "output_sha256":{p.name:digest(p) for p in OUT.iterdir() if p.is_file()}}
(OUT/"pilot_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print(pd.DataFrame(summary["primary_results"])[["method","AP","ROC_AUC","precision","recall","F1"]].to_string(index=False))
print("PILOT: PASS — kod tamamlandı; bu ifade yüksek performans garantisi değildir.")
print("Gönderilecek dosya:",(OUT/"pilot_summary.json").resolve())
